In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Script complet de téléchargement d'images pour application de lecture syllabique.
Optimisé pour Pixabay avec gestion des accents et éviter les doublons.
"""

import requests
import os
import time
import unicodedata
import logging
from pathlib import Path

# ============= CONFIGURATION =============
# Remplace par ta clé ou crée un fichier .env avec PIXABAY_API_KEY=ton_code
PIXABAY_API_KEY = "54287996-4cc776e38e881ac48c247c35e" 
OUTPUT_DIR = "images_lecture_syllabique"
LANGUAGE = "fr"
IMAGE_SIZE = "webformatURL" 
SAFESEARCH = True
DELAY_BETWEEN_REQUESTS = 0.8  # Environ 75 requêtes/minute (limite Pixabay: 100)

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(message)s',
    handlers=[logging.FileHandler('download.log', encoding='utf-8'), logging.StreamHandler()]
)


# ============= LISTE COMPLÈTE (500 MOTS) =============
VOCABULAIRE = {
    "animaux": [
        "chat", "chien", "lapin", "vache", "cheval", "cochon", "mouton", "poule", "canard", "oiseau",
        "poisson", "tortue", "serpent", "souris", "lion", "tigre", "ours", "elephant", "girafe", "zebre",
        "singe", "hippopotame", "crocodile", "kangourou", "panda", "koala", "baleine", "dauphin", "requin", "pieuvre",
        "crabe", "crevette", "abeille", "papillon", "fourmi", "coccinelle", "araignee", "escargot", "grenouille", "herisson",
        "ecureuil", "renard", "loup", "hibou", "aigle", "perroquet", "pingouin", "chameau", "ane", "chevre",
        "rhinoceros", "autruche", "gorille", "marmotte", "castor", "chauve-souris", "cygne", "flamant rose", "meduse", "autruche",
        "mouche", "moustique", "libellule", "autruche", "pigeon", "dindon", "poussin", "taureau", "veau", "biche"
    ],
    "aliments": [
        "pomme", "banane", "poire", "orange", "fraise", "cerise", "raisin", "citron", "melon", "ananas",
        "tomate", "carotte", "salade", "pomme de terre", "haricot", "oignon", "poivron", "champignon", "broccoli", "aubergine",
        "pain", "croissant", "gateau", "chocolat", "bonbon", "glace", "sucre", "sel", "miel", "confiture",
        "oeuf", "lait", "fromage", "yaourt", "beurre", "viande", "poulet", "poisson", "jambon", "saucisson",
        "riz", "pates", "soupe", "pizza", "burger", "frites", "sandwich", "quiche", "jus d'orange", "eau",
        "avocat", "peche", "abricot", "kiwi", "noix", "noisette", "amande", "farine", "huile", "vinaigre",
        "biscuit", "tarte", "crepe", "gaufre", "puree", "petit pois", "citrouille", "radis", "ail", "olive"
    ],
    "maison_objets": [
        "table", "chaise", "lit", "armoire", "canape", "fauteuil", "etagere", "bureau", "meuble", "tapis",
        "porte", "fenetre", "rideau", "mur", "toit", "escalier", "garage", "balcon", "jardin", "cheminee",
        "lampe", "miroir", "horloge", "reveil", "vase", "cadre", "coussin", "couverture", "drap", "oreiller",
        "assiette", "verre", "bol", "tasse", "cuillere", "fourchette", "couteau", "poele", "casserole", "four",
        "frigo", "aspirateur", "balai", "poubelle", "cle", "serrure", "telephone", "ordinateur", "television", "radio",
        "savon", "shampoing", "serviette", "brosse a dents", "dentifrice", "peigne", "brosse", "baignoire", "douche", "lavabo",
        "fer a repasser", "machine a laver", "parapluie", "valise", "sac a dos", "portefeuille", "lunettes", "montre", "ciseaux", "colle",
        "marteau", "tournevis", "pinceau", "outil", "echelle", "arrosoir", "tondeuse", "cloture", "boite", "journal"
    ],
    "vetements": [
        "pantalon", "short", "jupe", "robe", "chemise", "t-shirt", "pull", "gilet", "veste", "manteau",
        "pyjama", "slip", "culotte", "maillot de bain", "chaussette", "chaussure", "botte", "basket", "sandale", "pantoufle",
        "chapeau", "bonnet", "casquette", "echarpe", "gant", "ceinture", "cravate", "bouton", "fermeture", "poche"
    ],
    "corps_humain": [
        "tete", "cheveux", "visage", "oeil", "nez", "bouche", "dent", "langue", "oreille", "cou",
        "epaule", "bras", "coude", "poignet", "main", "doigt", "pouce", "ongle", "poitrine", "ventre",
        "dos", "fesse", "jambe", "genou", "cheville", "pied", "talon", "orteil", "coeur", "squelette"
    ],
    "nature": [
        "soleil", "lune", "etoile", "ciel", "nuage", "pluie", "neige", "vent", "orage", "arc-en-ciel",
        "arbre", "fleur", "herbe", "feuille", "branche", "racine", "foret", "buisson", "sapin", "chene",
        "montagne", "colline", "rocher", "pierre", "sable", "terre", "boue", "grotte", "volcan", "desert",
        "riviere", "lac", "mer", "ocean", "vague", "plage", "ile", "cascade", "marais", "campagne",
        "champ", "prairie", "jardin", "parc", "chemin", "route", "horizon", "planete", "univers", "feu",
        "rose", "tulipe", "marguerite", "tournesol", "lavande", "cactus", "mousse", "roseau", "ble", "vigne"
    ],
    "ecole": [
        "ecole", "classe", "tableau", "bureau", "chaise", "cartable", "trousse", "crayon", "stylo", "gomme",
        "regle", "ciseaux", "colle", "taille-crayon", "cahier", "livre", "feuille", "papier", "classeur", "ardoise",
        "craie", "feutre", "pinceau", "peinture", "calculatrice", "dictionnaire", "bibliotheque", "gymnase", "recreation", "cantine"
    ],
    "transport": [
        "voiture", "camion", "bus", "car", "taxi", "moto", "velo", "trottinette", "train", "metro",
        "tramway", "avion", "helicoptere", "fusee", "bateau", "voilier", "peniche", "sous-marin", "tracteur", "ambulance",
        "pompier", "police", "roue", "volant", "moteur", "gare", "aeroport", "port", "quai", "rail"
    ],
    "verbes_actions": [
        "manger", "boire", "dormir", "courir", "marcher", "sauter", "danser", "chanter", "jouer", "lire",
        "ecrire", "dessiner", "peindre", "colorier", "decouper", "coller", "plier", "rire", "pleurer", "sourire",
        "crier", "parler", "ecouter", "regarder", "voir", "entendre", "toucher", "sentir", "gouter", "attraper",
        "lancer", "pousser", "tirer", "porter", "tenir", "ouvrir", "fermer", "monter", "descendre", "tomber",
        "grimper", "glisser", "nager", "plonger", "voler", "conduire", "pedaler", "skier", "patiner", "rouler",
        "balayer", "laver", "cuisiner", "brosser", "peigner", "habiller", "chercher", "trouver", "donner", "prendre"
    ],
    "metiers_lieux": [
        "docteur", "infirmier", "dentiste", "veterinaire", "pompier", "policier", "facteur", "boulanger", "boucher", "fermier",
        "jardinier", "cuisinier", "serveur", "maitre", "maitresse", "pilote", "marin", "chanteur", "danseur", "peintre",
        "hopital", "mairie", "poste", "banque", "magasin", "marche", "boulangerie", "pharmacie", "cinema", "theatre"
    ],
    "niveau_1_cv_son_a": [
        "papa", "tata", "lama", "sac", "lac", "rat", "chat", "plat", "bras", "camera",
        "sofa", "bocal", "bavoir", "lavabo", "ananas", "banane", "cabane", "salade", "tomate", "fac",
        "macaque", "matelas", "canard", "dinde", "navet", "cafe", "cacao", "boa", "ara", "panda",
        "puma", "tapis", "radis", "pyjama", "pirate", "parachute", "canape", "patate", "pate", "farine"
    ],
    "niveau_1_cv_son_o": [
        "moto", "velo", "domino", "robot", "polo", "piano", "radio", "video", "lasso", "loto",
        "dynamo", "bol", "col", "vol", "sol", "loup", "ours", "pot", "lot", "mot",
        "sot", "bout", "mou", "trou", "clou", "chou", "pou", "fou", "cou", "sou",
        "roti", "soda", "dos", "os", "judo", "karate", "gros", "trop", "flot", "croc"
    ],
    "niveau_1_cv_son_i_y": [
        "lit", "nid", "kiwi", "sirene", "fil", "vif", "riz", "tic", "pic", "dodo",
        "maki", "zebu", "biche", "caniche", "figue", "olive", "livre", "tigre", "vit", "dit",
        "rit", "pli", "cri", "gris", "prix", "riz", "tapis", "vis", "amis", "avis"
    ],
    "niveau_1_cv_son_u": [
        "lune", "mur", "jus", "bus", "fusee", "tortue", "rue", "vue", "nue", "grue",
        "cru", "dru", "lu", "pu", "su", "tu", "vu", "bu", "sur", "pur",
        "dur", "pull", "duc", "bulle", "jupe", "flute", "plume", "sucre", "lute", "mule"
    ],
    "niveau_1_cv_son_e_eu": [
        "bebe", "fee", "tele", "epee", "puree", "bidet", "pneu", "fer", "mer", "sel",
        "ver", "bec", "chef", "cerf", "oeuf", "boeuf", "neuf", "nez", "pas", "main",
        "pied", "tete", "joue", "ble", "cle", "pre", "the", "de", "ne", "verre"
    ],
    "niveau_1_consonnes_doubles": [
        "pomme", "balle", "botte", "carotte", "classe", "gomme", "nappe", "patte", "poubelle", "tasse",
        "terre", "liasse", "crasse", "bosse", "fosse", "brosse", "tresse", "presse", "fesse", "caisse",
        "graisse", "laisse", "chaise", "fraise", "chasse", "masse", "passe", "tache", "vache", "cache"
    ]
}
# ============= FONCTIONS UTILES =============

def clean_filename(text: str) -> str:
    """Enlève les accents et remplace les espaces par des underscores."""
    nfkd_form = unicodedata.normalize('NFKD', text)
    only_ascii = nfkd_form.encode('ASCII', 'ignore').decode('ASCII')
    return only_ascii.lower().replace(" ", "_").replace("'", "_")

def get_api_key():
    api_key = os.getenv('PIXABAY_API_KEY', PIXABAY_API_KEY)
    if not api_key or api_key == "VOTRE_CLE_API_ICI":
        raise ValueError("Erreur: Clé API Pixabay manquante !")
    return api_key

def search_image(mot: str, api_key: str):
    """Recherche une image sur Pixabay."""
    url = "https://pixabay.com/api/"
    params = {
        "key": api_key,
        "q": mot,
        "lang": LANGUAGE,
        "image_type": "photo",
        "safesearch": str(SAFESEARCH).lower(),
        "per_page": 3,
        "order": "popular"
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data['hits'][0] if data.get('totalHits', 0) > 0 else None
    except Exception as e:
        logging.error(f"Erreur API pour '{mot}': {e}")
        return None

def download_file(url: str, dest_path: Path):
    """Télécharge physiquement le fichier."""
    try:
        response = requests.get(url, stream=True, timeout=15)
        response.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception as e:
        logging.error(f"Erreur téléchargement : {e}")
        return False

# ============= LOGIQUE PRINCIPALE =============

def main():
    key = get_api_key()
    base_dir = Path(OUTPUT_DIR)
    base_dir.mkdir(exist_ok=True)

    total_mots = sum(len(mots) for mots in VOCABULAIRE.values())
    count = 0

    logging.info(f"Démarrage du téléchargement pour {total_mots} mots.")

    for theme, mots in VOCABULAIRE.items():
        theme_dir = base_dir / theme
        theme_dir.mkdir(exist_ok=True)

        for mot in mots:
            count += 1
            filename_base = clean_filename(mot)
            image_path = theme_dir / f"{filename_base}.jpg"
            meta_path = theme_dir / f"{filename_base}.json"

            # 1. Skip si déjà présent
            if image_path.exists():
                logging.info(f"[{count}/{total_mots}] '{mot}' déjà présent. Skip.")
                continue

            # 2. Recherche
            logging.info(f"[{count}/{total_mots}] Recherche de '{mot}'...")
            result = search_image(mot, key)
            
            if result:
                # 3. Téléchargement
                image_url = result.get(IMAGE_SIZE)
                if download_file(image_url, image_path):
                    # 4. Sauvegarde métadonnées
                    import json
                    meta_data = {
                        "mot": mot,
                        "theme": theme,
                        "pixabay_id": result.get('id'),
                        "author": result.get('user'),
                        "tags": result.get('tags')
                    }
                    with open(meta_path, 'w', encoding='utf-8') as f:
                        json.dump(meta_data, f, ensure_ascii=False, indent=4)
                    
                    logging.info(f"Succès : '{mot}' sauvegardé.")
                
                # Respecter les limites de l'API
                time.sleep(DELAY_BETWEEN_REQUESTS)
            else:
                logging.warning(f"Aucun résultat pour '{mot}'.")

    logging.info("Terminé ! Toutes les images disponibles ont été récupérées.")

if __name__ == "__main__":
    main()

2026-02-23 14:13:19,541 - [INFO] - Démarrage du téléchargement pour 690 mots.
2026-02-23 14:13:19,542 - [INFO] - [1/690] Recherche de 'chat'...
2026-02-23 14:13:20,982 - [INFO] - Succès : 'chat' sauvegardé.
2026-02-23 14:13:21,783 - [INFO] - [2/690] Recherche de 'chien'...
2026-02-23 14:13:22,928 - [INFO] - Succès : 'chien' sauvegardé.
2026-02-23 14:13:23,730 - [INFO] - [3/690] Recherche de 'lapin'...
2026-02-23 14:13:24,917 - [INFO] - Succès : 'lapin' sauvegardé.
2026-02-23 14:13:25,718 - [INFO] - [4/690] Recherche de 'vache'...
2026-02-23 14:13:26,889 - [INFO] - Succès : 'vache' sauvegardé.
2026-02-23 14:13:27,690 - [INFO] - [5/690] Recherche de 'cheval'...
2026-02-23 14:13:28,948 - [INFO] - Succès : 'cheval' sauvegardé.
2026-02-23 14:13:29,756 - [INFO] - [6/690] Recherche de 'cochon'...
2026-02-23 14:13:30,912 - [INFO] - Succès : 'cochon' sauvegardé.
2026-02-23 14:13:31,713 - [INFO] - [7/690] Recherche de 'mouton'...
2026-02-23 14:13:33,194 - [INFO] - Succès : 'mouton' sauvegardé.
2

## Images des sons (a, e, i, o, u, eu, consonnes doubles) — Pexels

Pour remplacer les images Pixabay des thèmes PS par des photos Pexels (souvent plus claires et adaptées aux enfants), exécuter la cellule ci-dessous.

**Prérequis :** clé API gratuite sur [pexels.com/api](https://www.pexels.com/api/). Définir `PEXELS_API_KEY` ou la variable d'environnement `PEXELS_API_KEY`.

Les images sont enregistrées dans `assets/images/` et les JSON dans `assets/data/` sont mis à jour avec `image_path` et `pexels_id`.

In [ ]:
# Re-téléchargement des images des SONS uniquement via Pexels (remplace Pixabay pour ces thèmes)
import requests
import os
import time
import json
import unicodedata
import logging
from pathlib import Path

PEXELS_API_KEY = os.getenv("PEXELS_API_KEY", "VOTRE_CLE_PEXELS_ICI")
ASSETS = Path("assets")
IMAGES_DIR = ASSETS / "images"
DATA_DIR = ASSETS / "data"
DELAY_PEXELS = 0.5  # respecter ~200 req/h

def _session():
    s = requests.Session()
    s.trust_env = False  # ignorer proxy (env + système)
    s.proxies = {"http": None, "https": None}
    return s

THÈMES_SONS = [
    "niveau_1_cv_son_a",
    "niveau_1_cv_son_e_eu",
    "niveau_1_cv_son_i_y",
    "niveau_1_cv_son_o",
    "niveau_1_cv_son_u",
    "niveau_1_consonnes_doubles",
]

# Même vocabulaire que la cellule principale (extrait des thèmes sons)
VOCAB_SONS = {
    "niveau_1_cv_son_a": [
        "papa", "tata", "lama", "sac", "lac", "rat", "chat", "plat", "bras", "camera",
        "sofa", "bocal", "bavoir", "lavabo", "ananas", "banane", "cabane", "salade", "tomate", "fac",
        "macaque", "matelas", "canard", "dinde", "navet", "cafe", "cacao", "boa", "ara", "panda",
        "puma", "tapis", "radis", "pyjama", "pirate", "parachute", "canape", "patate", "pate", "farine",
    ],
    "niveau_1_cv_son_o": [
        "moto", "velo", "domino", "robot", "polo", "piano", "radio", "video", "lasso", "loto",
        "dynamo", "bol", "col", "vol", "sol", "loup", "ours", "pot", "lot", "mot",
        "sot", "bout", "mou", "trou", "clou", "chou", "pou", "fou", "cou", "sou",
        "roti", "soda", "dos", "os", "judo", "karate", "gros", "trop", "flot", "croc",
    ],
    "niveau_1_cv_son_i_y": [
        "lit", "nid", "kiwi", "sirene", "fil", "vif", "riz", "tic", "pic", "dodo",
        "maki", "zebu", "biche", "caniche", "figue", "olive", "livre", "tigre", "vit", "dit",
        "rit", "pli", "cri", "gris", "prix", "riz", "tapis", "vis", "amis", "avis",
    ],
    "niveau_1_cv_son_u": [
        "lune", "mur", "jus", "bus", "fusee", "tortue", "rue", "vue", "nue", "grue",
        "cru", "dru", "lu", "pu", "su", "tu", "vu", "bu", "sur", "pur",
        "dur", "pull", "duc", "bulle", "jupe", "flute", "plume", "sucre", "lute", "mule",
    ],
    "niveau_1_cv_son_e_eu": [
        "bebe", "fee", "tele", "epee", "puree", "bidet", "pneu", "fer", "mer", "sel",
        "ver", "bec", "chef", "cerf", "oeuf", "boeuf", "neuf", "nez", "pas", "main",
        "pied", "tete", "joue", "ble", "cle", "pre", "the", "de", "ne", "verre",
    ],
    "niveau_1_consonnes_doubles": [
        "pomme", "balle", "botte", "carotte", "classe", "gomme", "nappe", "patte", "poubelle", "tasse",
        "terre", "liasse", "crasse", "bosse", "fosse", "brosse", "tresse", "presse", "fesse", "caisse",
        "graisse", "laisse", "chaise", "fraise", "chasse", "masse", "passe", "tache", "vache", "cache",
    ],
}

def clean_filename(text):
    nfkd = unicodedata.normalize("NFKD", text)
    ascii_only = nfkd.encode("ASCII", "ignore").decode("ASCII")
    return ascii_only.lower().replace(" ", "_").replace("'", "_")

def search_pexels(mot, api_key, session=None):
    session = session or _session()
    url = "https://api.pexels.com/v1/search"
    headers = {"Authorization": api_key}
    params = {"query": mot, "per_page": 3, "locale": "fr-FR"}
    try:
        r = session.get(url, headers=headers, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        photos = data.get("photos", [])
        return photos[0] if photos else None
    except Exception as e:
        logging.warning("Pexels API '%s': %s", mot, e)
        return None

def download_file(url, dest_path, session=None):
    session = session or _session()
    try:
        r = session.get(url, stream=True, timeout=15)
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception as e:
        logging.warning("Téléchargement %s: %s", dest_path, e)
        return False

logging.basicConfig(level=logging.INFO, format="%(message)s")

if PEXELS_API_KEY == "VOTRE_CLE_PEXELS_ICI":
    print("Définir PEXELS_API_KEY (ou variable d'environnement PEXELS_API_KEY).")
else:
    total = sum(len(mots) for mots in VOCAB_SONS.values())
    n = 0
    session = _session()
    for theme in THÈMES_SONS:
        mots = VOCAB_SONS.get(theme, [])
        (IMAGES_DIR / theme).mkdir(parents=True, exist_ok=True)
        data_theme = DATA_DIR / theme
        data_theme.mkdir(parents=True, exist_ok=True)
        for mot in mots:
            n += 1
            fbase = clean_filename(mot)
            image_path = IMAGES_DIR / theme / f"{fbase}.jpg"
            json_path = data_theme / f"{fbase}.json"
            print(f"[{n}/{total}] {theme} — {mot}...", end=" ")
            photo = search_pexels(mot, PEXELS_API_KEY, session)
            if not photo:
                print("aucun résultat Pexels")
                time.sleep(DELAY_PEXELS)
                continue
            url = photo.get("src", {}).get("medium") or photo.get("src", {}).get("large") or photo.get("src", {}).get("original")
            if not url:
                print("pas d'URL image")
                time.sleep(DELAY_PEXELS)
                continue
            if download_file(url, image_path, session):
                rel_image = f"assets/images/{theme}/{fbase}.jpg"
                meta = {}
                if json_path.exists():
                    with open(json_path, "r", encoding="utf-8") as f:
                        meta = json.load(f)
                meta["mot"] = mot
                meta["theme"] = theme
                meta["image_path"] = rel_image
                meta["pexels_id"] = photo.get("id")
                meta["photographer"] = photo.get("photographer")
                if "syllabes" not in meta:
                    meta["syllabes"] = [mot]
                if "level" not in meta:
                    meta["level"] = 1
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(meta, f, ensure_ascii=False, indent=4)
                print("OK (Pexels)")
            else:
                print("échec téléchargement")
            time.sleep(DELAY_PEXELS)
    print("Terminé. Photos fournies par Pexels (pexels.com).")